# ViZDoom My Way Home: PPO + Curiosity (ICM) with AgileRL

In this notebook, we're going to train a PPO agent with curiosity to beat the My Way Home scenario in ViZDoom, using only visual observations from the game!

My Way Home is a maze. Doomguy is dropped into a random room, and a green armor vest is always in the same one. The rooms are different colors, so the screen is basically the map. The official writeup of the scenario is [here](https://vizdoom.farama.org/environments/default/#my-way-home). The catch is that the reward is very sparse: nothing happens until he stumbles onto the vest, so an agent that's just wandering gets almost no signal to learn from.

That makes it the perfect map for curiosity. It's actually the same map Pathak et al. used to introduce the Intrinsic Curiosity Module (ICM) in [Curiosity-driven Exploration by Self-supervised Prediction](https://arxiv.org/abs/1705.05363). The idea is to reward the agent for being surprised. A small forward model tries to predict the feature embedding of the next frame from the current frame's embedding and the action taken. Wherever that prediction is bad, Doomguy hasn't seen enough of that part of the maze, and the error gets paid out as a bonus reward. As places become familiar they stop paying, which pushes him on to somewhere new.

We'll still be using AgileRL for PPO, since it has good implementations of the base algorithms. But its training loop has no hook for adding a reward on top of the environment's, so this notebook breaks the agent apart and runs its own rollout and update loop. AgileRL still provides the networks, the observation preprocessing, the evaluation, and the checkpoints.

This one uses the stock maze. The episode ends when he reaches the vest, or when time runs out.

It also tries to use the original Doom WADs if it can find them.

The notebook was designed to run in place in the repo, however it also supports running on Google Colab, with the option of using Google Drive to save artifacts.

# Setup

Install the pinned dependencies first.

In [ ]:
import sys

# Update the required system packages when running on Colab.
if "google.colab" in sys.modules:
    !apt-get update -qq
    !apt-get install -y -qq libopenal1 libopenal-dev > /dev/null

# Install the Python packages used throughout the notebook.
%pip install -q vizdoom==1.3.1 agilerl==2.8.4 gymnasium==1.3.0 imageio==2.38.0 imageio-ffmpeg==0.6.0 opencv-python==5.0.0.93 scipy==1.17.1 numpy==2.4.6 pandas==3.0.5 torch==2.13.0 matplotlib==3.11.1

# Imports

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import gymnasium
import imageio.v2 as imageio
import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


from agilerl.algorithms.ppo import PPO
from gymnasium.spaces import Box
from gymnasium.wrappers import FrameStackObservation, ResizeObservation, TransformObservation
from IPython.display import Image, display
from scipy.io import wavfile
from torch.nn.utils import clip_grad_norm_
from tqdm.auto import tqdm


import vizdoom as vzd
from vizdoom import gymnasium_wrapper  # registers the Vizdoom* gymnasium envs
from vizdoom.gymnasium_wrapper.base_gymnasium_env import VizdoomEnv

# Experiment Setup
The observation setup matches the other notebooks. The agent acts every 4 frames and sees the last 4 frames stacked at 84x84, which gives it a sense of motion. My Way Home has 5 buttons, and we allow only one at a time, so the actions are turn left, turn right, walk forward, step left, step right, and do nothing. We run 16 environments side by side like the other PPO notebooks, because every update learns from a fresh batch collected across all of them.

`EVAL_EPISODES` controls how many episodes we use to score each checkpoint. `EVAL_LAST` limits that scoring to the last few checkpoints. `RECORD_EPISODES` controls how many videos we save. Set `RECORD_CHECKPOINTS` to `"all"` if you want every generation, although converting all of them takes a while.

In [ ]:
ENV_NAME = "VizdoomMyWayHome-v1"
NUM_ENVS = 16
FRAME_SKIP = 4          # act every 4 frames
STACK_SIZE = 4          # last 4 frames, so the agent can see motion
OBS_SIZE = (84, 84)
MAX_BUTTONS_PRESSED = 1 # one button at a time

EVAL_EPISODES = 5
EVAL_LAST = 5           # score only the last few checkpoints for now
RECORD_EPISODES = 3
RECORD_CHECKPOINTS = "best"  # or "all"

All the path logic lives in the next few cells, so the rest of the notebook doesn't need to care whether it's running on Colab or locally.

On Colab, `USE_DRIVE` keeps the run in Google Drive so it survives the session. Otherwise, everything stays in the notebook's folder.

In [ ]:
USE_DRIVE = True # Colab only: keep the experiment (and every output) in Google Drive
EXPERIMENT = "my-way-home-ppo-icm-AgileRL"

if "google.colab" in sys.modules and USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    experiment_dir = Path("/content/drive/MyDrive/rl-epsilon/vizdoom-agilerl-experiments") / EXPERIMENT
elif "google.colab" in sys.modules:
    experiment_dir = Path("/content") / EXPERIMENT  # lost when the Colab session ends
else:
    experiment_dir = Path.cwd()  # local: the notebook's own folder
print("experiment:", experiment_dir)

ViZDoom ships with Freedoom. But if `DOOM2.WAD` is in the `WADS` folder, we use that instead. Always better to use the original assets in my opinion! The scenario is the stock maze, `my_way_home.cfg`, which ships with ViZDoom.

In [ ]:
USE_DOOM2_WAD = True  # use DOOM2.WAD if found, otherwise ViZDoom's bundled Freedoom
STOCK_SCENARIO = "my_way_home.cfg"  # bundled with ViZDoom

scenario_cfg = str(Path(vzd.scenarios_path) / STOCK_SCENARIO)
SCENARIO_NAME = Path(scenario_cfg).stem

wad = experiment_dir.parent / "WADS" / "DOOM2.WAD"
DOOM_WAD = str(wad) if USE_DOOM2_WAD and wad.is_file() else None
IWAD_NAME = Path(DOOM_WAD).name if DOOM_WAD else "freedoom2"
print("scenario:", scenario_cfg)
print("IWAD:", IWAD_NAME)

Each run gets its own timestamped folder. The best model, charts, and videos are copied to `artifacts`.

In [ ]:
run_dir = experiment_dir / "training-runs" / f"run-{datetime.now():%Y%m%d_%H%M%S}"
checkpoint_dir = run_dir / "checkpoints"
recording_dir = run_dir / "recordings"
artifacts_dir = experiment_dir / "artifacts"
model_dir = artifacts_dir / "model"
video_dir = artifacts_dir / "video"
figure_dir = artifacts_dir / "plots_and_figures"
for d in (checkpoint_dir, recording_dir, model_dir, video_dir, figure_dir):
    d.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "best_model.pt"
best_icm_path = model_dir / "best_icm.pt"
metrics_csv = figure_dir / "generation_metrics.csv"
results_path = artifacts_dir / "results.json"
print("run:", run_dir.name)

# Environment

Next, we'll build the environment used to train Doomguy.

My Way Home is a maze of colored rooms. Doomguy can turn, walk forward, or step sideways. He starts in a random room, facing a random direction, and the green vest is always in the same room. Reaching it is worth +1. Every tic alive costs 0.0001, and the episode times out after 2100 tics.

`make_env` turns each frame into an 84x84 grayscale image and stacks the last 4 frames so the network can see motion. The vest is already worth about 1, so the reward stays as the game gives it. The curiosity bonus gets added later, inside the training loop, rather than in a wrapper, because it depends on the agent's own networks.

Training, evaluation, and recording all use that same stock maze. We then create `NUM_ENVS` copies, each in its own process, so they can step at the same time.

In [ ]:
def make_env(config_file=None):
    """Stock maze. Pass a config file to load my_way_home.cfg directly."""
    kwargs = dict(
        render_mode="rgb_array",
        frame_skip=FRAME_SKIP,
        screen_resolution=vzd.ScreenResolution.RES_200X125,
        max_buttons_pressed=MAX_BUTTONS_PRESSED,
        treat_episode_timeout_as_truncation=False,
    )
    if DOOM_WAD:
        kwargs["doom_game_path"] = DOOM_WAD
    env = VizdoomEnv(config_file=config_file, **kwargs) if config_file else gymnasium.make(ENV_NAME, **kwargs)

    h, w, _ = env.observation_space["screen"].shape
    env = TransformObservation(
        env,
        lambda obs: (obs["screen"] @ np.array([0.299, 0.587, 0.114])).astype(np.uint8),
        Box(low=0, high=255, shape=(h, w), dtype=np.uint8),
    )  # grayscale
    env = ResizeObservation(env, OBS_SIZE)  # 84x84
    env = FrameStackObservation(env, stack_size=STACK_SIZE)  # last 4 frames
    return env

# Create the vectorized training environment.
env = gymnasium.vector.AsyncVectorEnv(
    [make_env] * NUM_ENVS, autoreset_mode=gymnasium.vector.AutoresetMode.SAME_STEP
)

# Agent
Now we'll build the PPO agent. It learns the policy directly, as a probability for each action, and a critic that estimates how well things are going so the actor can tell whether an action beat expectations. It collects a fresh batch, takes a few careful updates, then throws the batch away.

PPO collects `LEARN_STEP` steps of fresh experience, 2048 in total, or 128 from each of the 16 environments. It then makes `UPDATE_EPOCHS` passes over that batch in chunks of `BATCH_SIZE`. `CLIP_COEF` limits how far one update can move the policy away from the policy that collected the data, and `ENT_COEF` adds a small bonus for keeping the action probabilities spread out. These are the same starting values as the other PPO notebooks. We train for about 2,000,000 steps. Every 25 updates, 51,200 steps, make up one generation: AgileRL evaluates one episode in each of the 16 environments and we save a checkpoint, which gives us 40 to compare.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon GPU
else:
    device = "cpu"
print("device:", device)

INIT_HP = {
    "BATCH_SIZE": 256,
    "LR": 2.5e-4,
    "GAMMA": 0.99,             # how much future reward counts
    "GAE_LAMBDA": 0.95,        # how far ahead the advantage looks
    "LEARN_STEP": 2048,        # fresh steps across all envs before an update
    "UPDATE_EPOCHS": 4,        # passes over that batch
    "CLIP_COEF": 0.1,          # how far one update can move the policy
    "ENT_COEF": 0.01,          # bonus for keeping actions spread out
    "VF_COEF": 0.5,            # weight of the value loss
    "MAX_GRAD_NORM": 0.5,      # gradient clipping
    "MAX_STEPS": 2_048_000,
    "EVO_STEPS": 51_200,       # steps per generation, 25 updates
    "EVAL_LOOP": 1,            # one episode in each env at the end of a generation
}

The network follows the original Atari DQN design: three convolutional layers, then a 256-unit hidden layer for each head. The actor and critic share the convolutional encoder, and that encoder's output is the feature embedding the curiosity module works in. AgileRL defaults to a 64-dimensional embedding and caps it at 128, so we raise both to 256 to give the forward model something richer to predict. The paper used 288.

In [ ]:
FEATURE_DIM = 256  # size of the shared encoder's output, which is also ICM's feature space

net_config = {
    "encoder_config": {"channel_size": [32, 64, 64], "kernel_size": [8, 4, 3], "stride_size": [4, 2, 1]},  # 3 conv layers
    "head_config": {"hidden_size": [256]},  # smaller than the original Atari head
    "latent_dim": FEATURE_DIM,
    "max_latent_dim": FEATURE_DIM,
}

We still build the agent with AgileRL's `PPO` class, which gives us the actor, the critic, observation preprocessing, `test`, and checkpoint saving and loading. We just won't call its `learn`, since that's where we need to add the curiosity reward and loss.

In [ ]:
ppo_agent = PPO(
    observation_space=env.single_observation_space,
    action_space=env.single_action_space,
    net_config=net_config,
    batch_size=INIT_HP["BATCH_SIZE"],
    lr=INIT_HP["LR"],
    learn_step=INIT_HP["LEARN_STEP"],
    gamma=INIT_HP["GAMMA"],
    gae_lambda=INIT_HP["GAE_LAMBDA"],
    update_epochs=INIT_HP["UPDATE_EPOCHS"],
    clip_coef=INIT_HP["CLIP_COEF"],
    ent_coef=INIT_HP["ENT_COEF"],
    vf_coef=INIT_HP["VF_COEF"],
    max_grad_norm=INIT_HP["MAX_GRAD_NORM"],
    num_envs=NUM_ENVS,
    device=device,
)
N_ACTIONS = ppo_agent.action_space.n

## Breaking out the agent
`get_action` hides the feature embedding: it runs the encoder and both heads and hands back only the results. So we split that forward pass in two. `embed` runs the shared encoder and returns the feature embedding. `act` and `evaluate` run the policy and value heads on top of an embedding we already have, so a single encoder pass serves PPO and ICM.

With shared encoders, AgileRL copies the actor's encoder into the critic once at construction, and from then on it only ever uses the actor's encoder followed by the critic's head. We do the same thing here, so a checkpoint saved from this loop still plays correctly through AgileRL's `get_action`.

In [ ]:
def embed(obs):
    # Shared encoder features for a batch of raw uint8 observations.
    return ppo_agent.actor.extract_features(ppo_agent.preprocess_observation(obs))


def act(features):
    # Sample actions from the policy head; returns (actions, log_probs, values).
    actions, log_probs, _ = ppo_agent.actor.forward_head(features)
    values = ppo_agent.critic.forward_head(features).squeeze(-1)
    return actions, log_probs, values


def evaluate(features, actions):
    # Log probs and entropy of the given actions, plus values, for the PPO update.
    _, _, entropy = ppo_agent.actor.forward_head(features, sample=False)
    log_probs = ppo_agent.actor.action_log_prob(actions)
    values = ppo_agent.critic.forward_head(features).squeeze(-1)
    return log_probs, entropy, values

# Intrinsic Curiosity Module
ICM sits on top of the feature embedding and has two small networks.

- The **inverse model** looks at the embeddings of two consecutive frames and guesses which action took Doomguy from one to the other. Training it shapes the encoder to keep what the agent can affect, like walls moving past as he turns, and to ignore what it can't.
- The **forward model** takes the current embedding and the action, and predicts the next embedding. Its error is the curiosity reward.

The ICM loss mixes the two as \((1-\beta)\,L_I + \beta\,L_F\), with \(\beta = 0.2\) as in the paper. The forward model's inputs and target are detached from the encoder. Because the encoder is shared with the policy, letting the forward loss reach it would let the encoder make prediction easy by squashing the features together, which would also starve the policy. This way the encoder is shaped by the inverse loss and by PPO, and the forward model just learns on top of it.

In [ ]:
class ICM(nn.Module):
    def __init__(self, feature_dim, n_actions, hidden_size=256):
        super().__init__()
        self.n_actions = n_actions
        self.inverse_model = nn.Sequential(
            nn.Linear(2 * feature_dim, hidden_size), nn.ReLU(), nn.Linear(hidden_size, n_actions),
        )
        self.forward_model = nn.Sequential(
            nn.Linear(feature_dim + n_actions, hidden_size), nn.ReLU(), nn.Linear(hidden_size, feature_dim),
        )

    def forward(self, features, next_features, actions):
        # Returns (forward_error, inverse_logits); forward_error is per sample.
        inverse_logits = self.inverse_model(torch.cat([features, next_features], dim=-1))
        one_hot = F.one_hot(actions, self.n_actions).float()
        predicted = self.forward_model(torch.cat([features.detach(), one_hot], dim=-1))
        forward_error = 0.5 * (predicted - next_features.detach()).pow(2).sum(dim=-1)
        return forward_error, inverse_logits

`INTRINSIC_COEF` sets how much curiosity is worth next to the vest. The raw forward error has no natural scale, and it shrinks as the forward model learns, so we divide it by its running standard deviation first. Then `INTRINSIC_COEF` means roughly "curiosity per step". At 0.01, a hundred steps of fresh territory is worth about one vest. Set it to 0 to get a plain PPO baseline with the exact same loop.

`ICM_COEF` weights the ICM loss next to the PPO loss when they're added together. The paper scales the policy loss down by 0.1 instead, which works out to an ICM weight of about 10. We start at 1 because the encoder here is shared with the policy.

In [ ]:
ICM_HP = {
    "INTRINSIC_COEF": 0.01,    # curiosity reward per step, after normalizing; 0 turns it off
    "ICM_COEF": 1.0,           # weight of the ICM loss next to the PPO loss
    "FORWARD_WEIGHT": 0.2,     # beta: forward loss vs inverse loss inside ICM
    "HIDDEN_SIZE": 256,        # hidden layer of the inverse and forward models
}

icm = ICM(FEATURE_DIM, N_ACTIONS, ICM_HP["HIDDEN_SIZE"]).to(device)

One optimizer covers everything that learns: the actor (encoder and policy head), the critic's head, and ICM. The critic's own copy of the encoder is left out, since it's never used.

In [ ]:
params = [*ppo_agent.actor.parameters(), *ppo_agent.critic.head_net.parameters(), *icm.parameters()]
optimizer = torch.optim.Adam(params, lr=INIT_HP["LR"])

`RunningStd` tracks the standard deviation of every forward error we've seen, merging in one batch at a time.

In [ ]:
class RunningStd:
    def __init__(self):
        self.mean, self.var, self.count = 0.0, 1.0, 1e-4

    def update(self, x):
        batch_mean, batch_var, n = float(x.mean()), float(x.var()), x.size
        delta = batch_mean - self.mean
        total = self.count + n
        self.mean += delta * n / total
        self.var = (self.var * self.count + batch_var * n + delta**2 * self.count * n / total) / total
        self.count = total

    @property
    def std(self):
        return float(np.sqrt(self.var)) + 1e-8


forward_error_std = RunningStd()

# Training
This is the custom loop. Each update has four steps: collect a rollout, score it with curiosity, compute advantages, and run the PPO and ICM losses together.

## Collecting a rollout
`collect_rollout` steps all 16 environments `LEARN_STEP / NUM_ENVS` times and stores everything as arrays shaped `(steps, envs, ...)`. Observations stay as uint8 on the CPU and only move to the GPU a minibatch at a time.

ICM needs the frame each action actually led to. The vector env resets a finished environment in the same step, so `next_obs` already shows the new episode for that slot. The real last frame is in `info["final_obs"]`, and we swap it in. Otherwise the forward model would get paid for failing to predict a teleport to a random room.

In [ ]:
def collect_rollout(obs, scores, n_steps):
    shape = (n_steps, NUM_ENVS)
    rollout = {
        "obs": np.zeros((*shape, *obs.shape[1:]), dtype=np.uint8),
        "next_obs": np.zeros((*shape, *obs.shape[1:]), dtype=np.uint8),
        "actions": np.zeros(shape, dtype=np.int64),
        "log_probs": np.zeros(shape, dtype=np.float32),
        "values": np.zeros(shape, dtype=np.float32),
        "rewards": np.zeros(shape, dtype=np.float32),
        "dones": np.zeros(shape, dtype=np.float32),
    }
    finished_scores = []
    for t in range(n_steps):
        with torch.no_grad():
            actions, log_probs, values = act(embed(obs))
        actions = actions.cpu().numpy()
        next_obs, reward, terminated, truncated, info = env.step(actions)
        done = terminated | truncated

        real_next_obs = next_obs.copy()
        for i in np.flatnonzero(info.get("_final_obs", [])):
            real_next_obs[i] = info["final_obs"][i]

        rollout["obs"][t], rollout["next_obs"][t], rollout["actions"][t] = obs, real_next_obs, actions
        rollout["log_probs"][t], rollout["values"][t] = log_probs.cpu().numpy(), values.cpu().numpy()
        rollout["rewards"][t], rollout["dones"][t] = reward, done

        scores += reward
        for i in np.flatnonzero(done):
            finished_scores.append(scores[i])
            scores[i] = 0.0
        obs = next_obs

    with torch.no_grad():
        rollout["last_value"] = act(embed(obs))[2].cpu().numpy()
    return rollout, obs, scores, finished_scores

## Curiosity reward and advantages
`forward_errors` runs the whole rollout back through the encoder and forward model to get the prediction error at every step. It's computed once per rollout with the current networks, so every transition in the batch is scored by the same model.

`compute_gae` is standard generalized advantage estimation on the combined reward. We write it ourselves instead of using the rollout buffer's version, because AgileRL's masks each step with the next step's done flag. Its collector stores "this step ended the episode", so that lines up one step late and lets value leak across episode boundaries. Here `dones[t]` cuts the bootstrap right after the step that ended the episode.

In [ ]:
@torch.no_grad()
def forward_errors(rollout, chunk=1024):
    obs = rollout["obs"].reshape(-1, *rollout["obs"].shape[2:])
    next_obs = rollout["next_obs"].reshape(obs.shape)
    actions = torch.as_tensor(rollout["actions"].reshape(-1), device=device)
    errors = []
    for i in range(0, len(obs), chunk):
        error, _ = icm(embed(obs[i:i + chunk]), embed(next_obs[i:i + chunk]), actions[i:i + chunk])
        errors.append(error.cpu().numpy())
    return np.concatenate(errors).reshape(rollout["actions"].shape)


def compute_gae(rewards, values, dones, last_value, gamma, gae_lambda):
    advantages = np.zeros_like(rewards)
    running = np.zeros_like(last_value)
    for t in reversed(range(len(rewards))):
        next_value = last_value if t == len(rewards) - 1 else values[t + 1]
        not_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        running = delta + gamma * gae_lambda * not_done * running
        advantages[t] = running
    return advantages, advantages + values

## The update
`update` is PPO's clipped update with the ICM loss added on. Each minibatch embeds the current and next frames in one encoder pass. The current embedding feeds both PPO heads and ICM, and the next embedding feeds only ICM. Everything is summed into one loss and stepped together, so the inverse loss trains the shared encoder alongside the policy.

The policy and value losses are the same as AgileRL's, value clipping included. Advantages are normalized over the whole batch. We also log how often the inverse model guesses the action right. Chance is 1 in 6, and if it never climbs above that, the encoder isn't learning anything about movement.

In [ ]:
def update(rollout, advantages, returns):
    flat = lambda x: x.reshape(-1, *x.shape[2:])
    to_tensor = lambda x: torch.as_tensor(flat(x), dtype=torch.float32, device=device)  # MPS has no float64
    obs, next_obs = flat(rollout["obs"]), flat(rollout["next_obs"])
    actions = torch.as_tensor(flat(rollout["actions"]), device=device)
    old_log_probs, old_values = to_tensor(rollout["log_probs"]), to_tensor(rollout["values"])
    advantages = to_tensor(advantages)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    returns = to_tensor(returns)

    clip = INIT_HP["CLIP_COEF"]
    beta = ICM_HP["FORWARD_WEIGHT"]
    totals = {k: 0.0 for k in ("loss", "policy_loss", "value_loss", "entropy", "inverse_loss", "forward_loss", "inverse_accuracy")}
    n_minibatches = 0
    for _ in range(INIT_HP["UPDATE_EPOCHS"]):
        for idx in np.array_split(np.random.permutation(len(obs)), len(obs) // INIT_HP["BATCH_SIZE"]):
            features, next_features = embed(np.concatenate([obs[idx], next_obs[idx]])).chunk(2)
            mb_actions = actions[idx]
            log_probs, entropy, values = evaluate(features, mb_actions)

            ratio = torch.exp(log_probs - old_log_probs[idx])
            policy_loss = torch.max(
                -advantages[idx] * ratio, -advantages[idx] * torch.clamp(ratio, 1 - clip, 1 + clip)
            ).mean()
            clipped_values = old_values[idx] + torch.clamp(values - old_values[idx], -clip, clip)
            value_loss = 0.5 * torch.max((values - returns[idx]) ** 2, (clipped_values - returns[idx]) ** 2).mean()
            entropy = entropy.mean()

            forward_error, inverse_logits = icm(features, next_features, mb_actions)
            forward_loss = forward_error.mean()
            inverse_loss = F.cross_entropy(inverse_logits, mb_actions)
            icm_loss = (1 - beta) * inverse_loss + beta * forward_loss

            loss = (
                policy_loss
                + INIT_HP["VF_COEF"] * value_loss
                - INIT_HP["ENT_COEF"] * entropy
                + ICM_HP["ICM_COEF"] * icm_loss
            )
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(params, INIT_HP["MAX_GRAD_NORM"])
            optimizer.step()

            inverse_accuracy = (inverse_logits.argmax(-1) == mb_actions).float().mean()
            for k, v in (("loss", loss), ("policy_loss", policy_loss), ("value_loss", value_loss), ("entropy", entropy),
                         ("inverse_loss", inverse_loss), ("forward_loss", forward_loss), ("inverse_accuracy", inverse_accuracy)):
                totals[k] += v.item()
            n_minibatches += 1
    return {k: v / n_minibatches for k, v in totals.items()}

## Running it
Each generation runs 25 updates. At the end, AgileRL's `test` plays one episode in each environment without the curiosity bonus, so `eval_fitness` is the real game score. We then save the agent with AgileRL's `save_checkpoint` and save ICM's weights next to it. `test` leaves the environments mid-episode, so we reset them before the next generation, the same way AgileRL's own loop does.

Each generation logs one row: the PPO and ICM losses, the inverse model's accuracy, the mean curiosity reward per step, the mean extrinsic score of the training episodes that finished, and the evaluation fitness.

A run this long is a good time to step away from the machine.

In [ ]:
history = []
n_steps = INIT_HP["LEARN_STEP"] // NUM_ENVS
updates_per_generation = INIT_HP["EVO_STEPS"] // INIT_HP["LEARN_STEP"]
total_steps = 0

obs, _ = env.reset()
scores = np.zeros(NUM_ENVS)
pbar = tqdm(total=INIT_HP["MAX_STEPS"], unit="step")
for generation in range(INIT_HP["MAX_STEPS"] // INIT_HP["EVO_STEPS"]):
    ppo_agent.set_training_mode(True)
    update_metrics, episode_scores, intrinsic = [], [], []
    for _ in range(updates_per_generation):
        rollout, obs, scores, finished = collect_rollout(obs, scores, n_steps)
        episode_scores += finished

        errors = forward_errors(rollout)
        forward_error_std.update(errors)
        intrinsic_rewards = ICM_HP["INTRINSIC_COEF"] * errors / forward_error_std.std
        intrinsic.append(intrinsic_rewards.mean())

        advantages, returns = compute_gae(
            rollout["rewards"] + intrinsic_rewards, rollout["values"], rollout["dones"],
            rollout["last_value"], INIT_HP["GAMMA"], INIT_HP["GAE_LAMBDA"],
        )
        update_metrics.append(update(rollout, advantages, returns))
        total_steps += n_steps * NUM_ENVS
        pbar.update(n_steps * NUM_ENVS)

    eval_fitness = ppo_agent.test(env, loop=INIT_HP["EVAL_LOOP"])
    obs, _ = env.reset()
    scores[:] = 0.0

    row = {"steps": total_steps, **pd.DataFrame(update_metrics).mean().to_dict()}
    row["intrinsic_reward"] = float(np.mean(intrinsic))
    row["train_score"] = float(np.mean(episode_scores)) if episode_scores else np.nan
    row["eval_fitness"] = eval_fitness
    history.append(row)
    pbar.set_postfix(eval=f"{eval_fitness:.2f}", inv_acc=f"{row['inverse_accuracy']:.2f}")

    ppo_agent.save_checkpoint(str(checkpoint_dir / f"PPO_{total_steps}.pt"))
    torch.save(icm.state_dict(), checkpoint_dir / f"ICM_{total_steps}.pt")
pbar.close()
env.close()

# Evaluation
During training, AgileRL's `test` scores each generation with one episode per environment. Afterward we reload checkpoints and play more episodes, but only the last `EVAL_LAST` of them for now. That's enough to see whether the end of the run is reaching the vest. Evaluation is the plain game reward, with no curiosity bonus.

`load_agent` loads a checkpoint, while `play` runs one episode and returns its total reward and action history. ICM isn't needed to play, only to train. Like the other PPO notebooks, we keep sampling from the policy at evaluation instead of always taking the most likely action, which can leave him stuck turning in place.

In [ ]:
def load_agent(path):
    agent = PPO.load(str(path), device=device)
    agent.set_training_mode(False)
    return agent


def play(env, agent, obs):
    # Run one episode from `obs`; returns (total_reward, actions).
    total, actions, done = 0.0, [], False
    while not done:
        action = int(agent.get_action(obs[None])[0][0])
        obs, reward, terminated, truncated, _ = env.step(action)
        total += reward
        actions.append(action)
        done = terminated or truncated
    return total, actions

The step count is at the end of each filename, so sorting by that value places the checkpoints in generation order. The ICM weights sit next to them with an `ICM_` prefix.

In [ ]:
checkpoints = sorted(checkpoint_dir.glob("PPO_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
print(f"found {len(checkpoints)} checkpoints")

We play `EVAL_EPISODES` episodes on the last `EVAL_LAST` checkpoints and record the rewards. The checkpoint with the highest average wins; ties go to the later generation. The selected checkpoint and its ICM weights are then copied to `artifacts`.

In [ ]:
eval_env = make_env(scenario_cfg)
eval_from = max(0, len(checkpoints) - EVAL_LAST)
eval_rewards = [None] * len(checkpoints)
best_gen = eval_from
for gen, ckpt in enumerate(checkpoints[eval_from:], start=eval_from):
    agent = load_agent(ckpt)
    rewards = []
    for episode in range(EVAL_EPISODES):
        obs, _ = eval_env.reset()
        reward, _ = play(eval_env, agent, obs)
        rewards.append(reward)
    eval_rewards[gen] = rewards
    mean = np.mean(rewards)
    print(f"gen {gen:>3}  mean={mean:6.2f}  rewards={np.round(rewards, 2).tolist()}")
    if mean >= np.mean(eval_rewards[best_gen]):  # ties go to the later generation
        best_gen = gen
eval_env.close()

best_checkpoint = checkpoints[best_gen]
shutil.copy2(best_checkpoint, best_model_path)
shutil.copy2(best_checkpoint.with_name(best_checkpoint.name.replace("PPO_", "ICM_")), best_icm_path)
print(f"best: gen {best_gen} ({best_checkpoint.name}), mean eval reward {np.mean(eval_rewards[best_gen]):.2f}")

# Analysis
Let's see how it did! Training is noisy and the eval episodes aren't seeded, so your curves won't match these exactly. The notes below are about the general shape.

## Metrics
One row per generation: the PPO losses and entropy, the ICM losses and inverse accuracy, the mean curiosity reward, the training score, AgileRL's evaluation fitness, and our checkpoint evaluation's mean and standard deviation. Checkpoint columns are filled in for the last few generations. The full table gets saved as a CSV in `artifacts`. Here we just show the first few generations, the best one, and the last few.

In [ ]:
metrics = pd.DataFrame(history)
metrics.index.name = "generation"
metrics["ckpt_eval_mean"] = [np.mean(r) if r is not None else np.nan for r in eval_rewards]
metrics["ckpt_eval_std"] = [np.std(r) if r is not None else np.nan for r in eval_rewards]
shown = pd.concat([metrics.head(3), metrics.loc[[best_gen]], metrics.tail(3)])
shown[~shown.index.duplicated()]

In [ ]:
metrics.to_csv(metrics_csv)

## Charts
Saved to `artifacts/plots_and_figures`, and shown here too.

In [ ]:
def save(fig, name):
    fig.savefig(figure_dir / name, dpi=150, bbox_inches="tight")
    plt.show()


gens = metrics.index

**Loss.** PPO's losses don't say much about how well the agent plays. The value loss is how far off the critic is, and with curiosity in the reward the critic is chasing a moving target, so expect it to stay noisy. The policy loss stays near zero by design, since clipping keeps every update small.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["value_loss"], marker="o")
ax1.set(xlabel="Generation", ylabel="Mean value loss", title="Critic (value) loss per generation")
ax2.plot(gens, metrics["policy_loss"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean policy loss", title="Actor (policy) loss per generation")
save(fig, "loss.png")

**ICM.** The inverse accuracy is the health check for the feature embedding. It should climb well above chance, the dashed line, as the encoder learns what Doomguy's actions do to the view. The forward loss is how surprised the forward model is on average. It usually drops quickly, then levels off or creeps back up as he reaches parts of the maze it hasn't seen.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["inverse_accuracy"], marker="o")
ax1.axhline(1 / N_ACTIONS, color="gray", linestyle="--", label="Chance")
ax1.set(xlabel="Generation", ylabel="Accuracy", title="Inverse model accuracy per generation")
ax1.legend()
ax2.plot(gens, metrics["forward_loss"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean forward loss", title="Forward model loss per generation")
save(fig, "icm_loss.png")

**Curiosity.** The left chart is the curiosity bonus paid out per step. The right one is the real game score of the training episodes, with no bonus included. Curiosity should carry the agent through the early generations while the score sits near the timeout. Once he starts finding the vest, the score should climb while the bonus fades, because the route he takes has become familiar.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["intrinsic_reward"], marker="o")
ax1.set(xlabel="Generation", ylabel="Mean reward per step", title="Curiosity reward per generation")
ax2.plot(gens, metrics["train_score"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean episode score", title="Training score (extrinsic only)")
save(fig, "intrinsic_reward.png")

**Entropy.** Entropy measures how spread out the action probabilities are. It starts near the maximum, the dashed line where every action is equally likely, and should fall as the agent gets more confident. If it crashes toward zero early, the agent stopped exploring too soon.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, metrics["entropy"], marker="o")
ax.axhline(np.log(N_ACTIONS), color="gray", linestyle="--", label="Maximum (uniform policy)")
ax.set(xlabel="Generation", ylabel="Mean entropy", title="Policy entropy per generation")
ax.legend()
save(fig, "entropy.png")

**Average reward.** A found vest lands near 1. A timeout lands near -0.2, because every tic costs a tiny bit and the episode is long. Blue shows AgileRL's evaluation during training. Orange shows our checkpoint evaluation on the last few generations, with the shaded band representing one standard deviation.

The band stays wide while he's still wandering, since one vest and one timeout pull the average in opposite directions. Orange only covers the end of the run. If it sits near 1, he found the vest.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, metrics["eval_fitness"], marker="o", label=f"Training eval ({INIT_HP['EVAL_LOOP'] * NUM_ENVS} episodes, {ENV_NAME})")
(line,) = ax.plot(gens, metrics["ckpt_eval_mean"], marker="o", label=f"Checkpoint eval ({EVAL_EPISODES} episodes, {SCENARIO_NAME})")
ax.fill_between(gens, metrics["ckpt_eval_mean"] - metrics["ckpt_eval_std"],
                metrics["ckpt_eval_mean"] + metrics["ckpt_eval_std"], color=line.get_color(), alpha=0.2)
ax.axvline(best_gen, color="gray", linestyle="--", label=f"Best generation ({best_gen})")
ax.set(xlabel="Generation", ylabel="Average reward", title="Average reward per generation")
ax.legend()
save(fig, "average_reward.png")

# Recording
To record the trained agent, we save ViZDoom's `.lmp` demos—which store only the inputs—and then replay them at 720p.

The Gymnasium wrapper doesn't expose demo recording, so `reset_with_lmp` temporarily replaces the reset function with one that starts a recording. `flush_lmp` begins the next episode so the file is actually written.

One gotcha: ViZDoom hangs if the demo path is longer than about 112 characters. We pass a path relative to the working directory, which keeps it short locally. Paths are longer on Colab when using Drive, so recording the best model still works, but `RECORD_CHECKPOINTS = "all"` can hang. Set `USE_DRIVE = False` if you want a video of every generation on Colab.

In [ ]:
def reset_with_lmp(env, lmp_path):
    base = env.unwrapped
    orig_reset = base.reset

    def recorded_reset(*, seed=None, options=None):
        gymnasium.Env.reset(base, seed=seed)
        game_seed = int(
            base.np_random.integers(0, np.iinfo(np.uint32).max + 1, dtype=np.uint32)
        )
        base.game.set_seed(game_seed)
        if not base.game.is_running():
            base.game.init()
        # ViZDoom hangs if the demo path is longer than ~112 characters, so pass it relative to the working directory
        base.game.new_episode(os.path.relpath(lmp_path))
        base.state = base.game.get_state()
        return base._VizdoomEnv__collect_observations(), {}

    base.reset = recorded_reset
    try:
        return env.reset()
    finally:
        base.reset = orig_reset


def flush_lmp(env):
    env.unwrapped.game.new_episode()

We record `RECORD_EPISODES` episodes with the best checkpoint, or with every checkpoint when `RECORD_CHECKPOINTS` is set to `"all"`. Best-model videos go in `artifacts/video`; all others remain in the run folder. Existing best-model videos are cleared first so they don't get mixed with the current run.

In [ ]:
assert RECORD_CHECKPOINTS in ("best", "all")
record_checkpoints = [best_checkpoint] if RECORD_CHECKPOINTS == "best" else checkpoints

for old in video_dir.glob("best_episode_*"):
    old.unlink()

In [ ]:
record_env = make_env(scenario_cfg)
lmps = []
lmp_rewards = {}
for ckpt in record_checkpoints:
    gen = checkpoints.index(ckpt)
    out_dir, prefix = (video_dir, "best") if ckpt == best_checkpoint else (recording_dir, f"gen_{gen}")
    agent = load_agent(ckpt)
    for episode in range(RECORD_EPISODES):
        lmp = out_dir / f"{prefix}_episode_{episode}.lmp"
        obs, _ = reset_with_lmp(record_env, lmp)
        reward, _ = play(record_env, agent, obs)
        print(f"gen {gen:>2}  episode {episode}  reward={reward:.2f}  -> {lmp.name}")
        lmps.append(lmp)
        lmp_rewards[lmp] = reward
flush_lmp(record_env)
record_env.close()

`lmp_to_mp4` replays each demo at 720p with the HUD enabled. The result looks much better than the 84x84 input used for training because we're replaying the actions in a fresh game rather than saving what the network saw. CRF 18, tuned for Doom's flat textures, keeps the files small enough for GitHub. Audio capture is silent on my Mac, so locally generated videos have no sound. Encoding takes a while because the encoder is configured to prioritize compression quality over speed.

In [ ]:
SAMPLE_RATE = 22050
VIDEO_PARAMS = dict(  # x264 at constant quality: CRF 18 is visually lossless, and "animation" suits Doom's flat textures
    fps=35, codec="libx264", quality=None, macro_block_size=8,
    ffmpeg_params=["-crf", "18", "-preset", "veryslow", "-tune", "animation", "-movflags", "+faststart"],
)


def lmp_to_mp4(lmp_path):
    """Replay an .lmp at 720p with HUD and audio, and write it next to the .lmp as .mp4."""
    lmp_path = Path(lmp_path)
    mp4_path = lmp_path.with_suffix(".mp4")
    silent_path = lmp_path.with_name(lmp_path.stem + "_silent.mp4")
    wav_path = lmp_path.with_suffix(".wav")

    game = vzd.DoomGame()
    game.load_config(scenario_cfg)
    if DOOM_WAD:
        game.set_doom_game_path(DOOM_WAD)
    game.set_screen_resolution(vzd.ScreenResolution.RES_1280X720)
    game.set_render_hud(True)
    game.set_render_corpses(True)
    game.set_window_visible(False)
    game.set_audio_buffer_enabled(True)
    game.set_audio_sampling_rate(vzd.SamplingRate.SR_22050)
    game.set_audio_buffer_size(1)
    game.add_game_args("+snd_efx 0")
    game.set_mode(vzd.Mode.SPECTATOR)
    game.init()

    frames, audio = [], []
    try:
        game.replay_episode(os.path.relpath(lmp_path))  # relative for the same path length limit as recording
        while not game.is_episode_finished():
            state = game.get_state()
            if state is not None:
                frames.append(np.transpose(state.screen_buffer, (1, 2, 0)))
                if state.audio_buffer is not None:
                    audio.append(state.audio_buffer)
            game.advance_action()
    finally:
        game.close()

    imageio.mimsave(silent_path, frames, **VIDEO_PARAMS)

    audio = np.concatenate(audio) if audio else np.zeros(1)
    if not audio.any():  # no sound captured (e.g. on macOS): keep the silent video
        silent_path.replace(mp4_path)
        return mp4_path

    wavfile.write(wav_path, SAMPLE_RATE, audio)
    subprocess.run(
        [imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", silent_path, "-i", wav_path,
         "-c:v", "copy", "-c:a", "aac", "-shortest", mp4_path],
        check=True, capture_output=True,
    )
    silent_path.unlink()
    wav_path.unlink()
    return mp4_path

In [ ]:
mp4s = [lmp_to_mp4(lmp) for lmp in lmps]
mp4s

For a quick preview, the highest-scoring episode is converted to a deliberately small GIF that plays directly in the notebook and on GitHub. The full-resolution MP4s remain in `artifacts/video`.

In [ ]:
GIF_WIDTH = 240
GIF_FPS = 10


def mp4_to_gif(mp4_path):
    """Write a small GIF next to an MP4, with a palette built from the video so the colors hold up."""
    gif_path = Path(mp4_path).with_suffix(".gif")
    vf = (
        f"fps={GIF_FPS},scale={GIF_WIDTH}:-1:flags=lanczos,split[a][b];"
        "[a]palettegen=max_colors=128:stats_mode=diff[p];[b][p]paletteuse=dither=bayer:bayer_scale=5:diff_mode=rectangle"
    )
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", mp4_path, "-vf", vf, gif_path], check=True, capture_output=True)
    return gif_path


best_mp4 = max(
    (m for m in mp4s if m.name.startswith("best")),
    key=lambda m: lmp_rewards[m.with_suffix(".lmp")],
)
best_gif = mp4_to_gif(best_mp4)
print(f"{best_gif.name}  reward={lmp_rewards[best_mp4.with_suffix('.lmp')]:.2f}  ({best_gif.stat().st_size / 2**20:.1f} MB)")
display(Image(filename=str(best_gif)))

# Results
A short summary of the run, also saved to `artifacts/results.json` so runs are easy to compare later.

In [ ]:
results = {
    "run": run_dir.name,
    "scenario": SCENARIO_NAME,
    "iwad": IWAD_NAME,
    "best_checkpoint": best_checkpoint.name,
    "best_generation": best_gen,
    "best_steps": int(metrics.loc[best_gen, "steps"]),
    "generations": len(checkpoints),
    "checkpoint_eval_mean": round(float(metrics.loc[best_gen, "ckpt_eval_mean"]), 4),
    "checkpoint_eval_std": round(float(metrics.loc[best_gen, "ckpt_eval_std"]), 4),
    "checkpoint_eval_rewards": [round(float(r), 4) for r in eval_rewards[best_gen]],
    "best_video": [str(m.relative_to(experiment_dir)) for m in mp4s if m.name.startswith("best")],
    "best_gif": str(best_gif.relative_to(experiment_dir)),
    "best_icm": str(best_icm_path.relative_to(experiment_dir)),
    "icm_hp": ICM_HP,
}
results_path.write_text(json.dumps(results, indent=2))
pd.Series(results, name="value").to_frame()

# Wrapping Up
My Way Home is where plain exploration struggles. Doomguy has to recognize the rooms and walk to the vest from a random start, and the only reward shows up when he gets there. Curiosity gives him something to chase in the meantime: every unfamiliar room pays out until it isn't unfamiliar anymore.

The interesting comparison is against the same loop with `INTRINSIC_COEF = 0`, which is plain PPO with everything else held fixed. If the curiosity run finds the vest and the baseline doesn't, ICM did its job. The ICM charts are the first place to look if it doesn't: an inverse accuracy stuck at chance means the feature embedding never learned anything useful to be curious about.

Happy Coding!